### Libraries

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.messages import HumanMessage
from langchain.agents import create_agent
import os
import requests

In [6]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="C:\\Users\\nasso\\Desktop\\langchain\\lca-lc-foundations\\example.env", override=True)

True

### Tools

In [ ]:
@tool
def get_current_weather(city: str) -> str:
    """
    Fetch the current weather conditions for a given city name using OpenWeatherMap.
    
    Args:
        city (str): The name of the city (e.g., 'Hyderabad', 'London').
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        return "Error: OPENWEATHER_API_KEY environment variable is not set."

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"  # Returns temperature in Celsius
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if response.status_code == 404:
            return f"Error: City '{city}' not found"
        elif response.status_code != 200:
            return f"Error: Failed to retrieve data ({data.get('message', 'Unknown error')})."

        temp = data["main"]["temp"]
        feels_like = data["main"]["feels_like"]
        condition = data["weather"][0]["description"]
        humidity = data["main"]["humidity"]
        wind_speed = data["wind"]["speed"]

        return (
            f"Weather in {city.title()}:\n"
            f"- Temperature: {temp}°C (feels like {feels_like}°C)\n"
            f"- Condition: {condition}\n"
            f"- Humidity: {humidity}%"
            f"- Wind Speed: {wind_speed} m/s"
        )

    except requests.exceptions.RequestException as e:
        return f"Network error while fetching weather data: {str(e)}"

### Model configuration

In [30]:
system_prompt = """ 
You are a weather telling agent. The user will give you a city name and you will return the current weather conditions for that city and clothing suggestions.
Do not mention anything else.
"""

In [31]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_current_weather],
    system_prompt=system_prompt
)

In [32]:
def get_weather(city: str) -> str:
    """
    Get the current weather for a specified city using the agent.
    
    Args:
        city (str): The name of the city (e.g., 'Athens').
    """

    return agent.invoke(
        {"messages": [HumanMessage(content=f"What is the weather like in {city}?")]}
    )

In [33]:
print(get_weather("Athens")["messages"][-1].content)

Weather in Athens:
- Temperature: 36.96°C (feels like 37.69°C)
- Condition: Clear sky
- Humidity: 30%
- Wind: 7.2 m/s

Clothing suggestions:
- Lightweight, breathable clothing (cotton or linen) in light colors
- Shorts or airy trousers with a light T-shirt or blouse
- Wide-brimmed hat or cap and sunglasses
- Sunscreen (SPF 30+)
- Comfortable sandals or breathable shoes
- Stay hydrated and seek shade during peak sun hours


In [34]:
print(get_weather("efefwef")["messages"][-1].content)

I couldn’t find a city named "efefwef." Please provide a valid city name for the current weather and clothing suggestions.
